# 🏒 Hockey Multi-Object Tracking with McByte

This notebook demonstrates how to use the optimized McByte tracker with your custom YOLO hockey detection model.

## Features
- **Custom YOLO Integration**: Works with your trained hockey detection model
- **McByte Tracking**: ByteTrack-based tracking with mask propagation
- **Cutie + SAM**: Temporal mask propagation for robust association
- **Production Ready**: Optimized, modular code structure

## Your Hockey Classes
```python
CLASS_NAMES = {
    0: "Center Ice",
    1: "Faceoff",
    2: "Goalpost",
    3: "Goaltender",
    4: "Player",
    5: "Puck",
    6: "Referee"
}
# Tracked classes: Player (4), Puck (5), Referee (6), Goaltender (3)
```

## 1. Installation & Setup

In [ ]:
# Install required packages (run once)
# !pip install ultralytics torch torchvision opencv-python scipy
# !pip install git+https://github.com/facebookresearch/segment-anything.git

# For Cutie (mask propagation) - clone and install
# !git clone https://github.com/hkchengrex/Cutie.git
# !cd Cutie && pip install -e .

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path('.').absolute()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import cv2
import torch
from IPython.display import display, Video, Image
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

Set up your tracking parameters and model paths.

In [ ]:
from configs.config import (
    PipelineConfig, 
    DetectorConfig, 
    TrackerConfig, 
    MaskConfig,
    VisualizationConfig,
    TrackableClass
)

# ==========================================
# CONFIGURE YOUR SETTINGS HERE
# ==========================================

# Path to your trained YOLO model
YOLO_MODEL_PATH = "weights/hockey_yolo.pt"  # <-- UPDATE THIS

# Path to Cutie weights (for mask propagation)
CUTIE_WEIGHTS = "weights/cutie-base-mega.pth"  # <-- UPDATE THIS

# Path to SAM weights
SAM_WEIGHTS = "weights/sam_vit_b_01ec64.pth"  # <-- UPDATE THIS

# Input video/images
INPUT_PATH = "data/hockey_game.mp4"  # <-- UPDATE THIS

# Output path
OUTPUT_PATH = "outputs/tracked_hockey.mp4"

# Hockey class configuration
CLASS_NAMES = {
    0: "Center Ice",
    1: "Faceoff",
    2: "Goalpost",
    3: "Goaltender",
    4: "Player",
    5: "Puck",
    6: "Referee"
}

# Classes to track (use class IDs)
TRACK_CLASSES = [
    TrackableClass.PLAYER.value,      # 4
    TrackableClass.PUCK.value,        # 5
    TrackableClass.REFEREE.value,     # 6
    TrackableClass.GOALTENDER.value,  # 3
]

# Class colors (BGR format)
CLASS_COLORS = {
    3: (255, 0, 0),      # Goaltender - Blue
    4: (0, 255, 0),      # Player - Green  
    5: (0, 255, 255),    # Puck - Yellow
    6: (0, 0, 255),      # Referee - Red
}

print("Configuration set!")
print(f"Tracking classes: {[CLASS_NAMES[c] for c in TRACK_CLASSES]}")

In [ ]:
# Create configuration object
config = PipelineConfig()

# Detector config
config.detector.model_path = YOLO_MODEL_PATH
config.detector.track_classes = TRACK_CLASSES
config.detector.class_names = CLASS_NAMES
config.detector.confidence_threshold = 0.25
config.detector.nms_threshold = 0.45
config.detector.device = "cuda:0" if torch.cuda.is_available() else "cpu"
config.detector.half_precision = torch.cuda.is_available()

# Tracker config
config.tracker.track_thresh = 0.5      # High confidence threshold
config.tracker.track_buffer = 30       # Keep lost tracks for 30 frames
config.tracker.match_thresh = 0.8      # IoU threshold
config.tracker.low_thresh = 0.1        # Low confidence threshold
config.tracker.new_track_thresh = 0.6  # Min confidence for new tracks
config.tracker.frame_rate = 30         # Video FPS
config.tracker.use_gmc = True          # Enable camera motion compensation

# Mask config
config.mask.cutie_weights = CUTIE_WEIGHTS
config.mask.sam_weights = SAM_WEIGHTS
config.mask.device = config.detector.device
config.mask.max_internal_size = 480    # Lower for faster processing

# Visualization config
config.visualization.show_boxes = True
config.visualization.show_ids = True
config.visualization.show_masks = True
config.visualization.show_trajectories = True
config.visualization.trajectory_length = 30
config.visualization.class_colors = CLASS_COLORS

print("Configuration created!")
print(f"Device: {config.detector.device}")

## 3. Initialize Components

Load the detector, tracker, and mask propagator.

In [ ]:
from detectors.yolo_detector import create_detector, UltralyticsYOLODetector

# Create detector
detector = create_detector(config.detector)

# Warm up
detector.warmup()

print("Detector ready!")

In [ ]:
from tracking.mcbyte_tracker import McByteTracker
from tracking.track import Track

# Reset track IDs
Track.reset_id()

# Create tracker
tracker = McByteTracker(
    track_thresh=config.tracker.track_thresh,
    track_buffer=config.tracker.track_buffer,
    match_thresh=config.tracker.match_thresh,
    low_thresh=config.tracker.low_thresh,
    new_track_thresh=config.tracker.new_track_thresh,
    frame_rate=config.tracker.frame_rate,
    use_gmc=config.tracker.use_gmc,
)

print("Tracker ready!")

In [ ]:
# Optional: Initialize mask propagator (for mask-enhanced tracking)
USE_MASKS = False  # Set to True if you have Cutie+SAM installed

mask_propagator = None
if USE_MASKS:
    try:
        from mask_propagation.mask_propagator import create_mask_propagator
        mask_propagator = create_mask_propagator(config.mask)
        print("Mask propagator ready!")
    except Exception as e:
        print(f"Mask propagator not available: {e}")
        print("Continuing without mask propagation...")
        USE_MASKS = False

## 4. Test on Single Frame

In [ ]:
from utils.visualization import Visualizer, draw_detections

# Create visualizer
visualizer = Visualizer(
    show_boxes=True,
    show_ids=True,
    show_trajectories=True,
    class_colors=CLASS_COLORS,
    class_names=CLASS_NAMES,
)

In [ ]:
# Load a test frame
cap = cv2.VideoCapture(INPUT_PATH)
ret, test_frame = cap.read()
cap.release()

if ret:
    print(f"Frame shape: {test_frame.shape}")
    
    # Run detection
    detections = detector.detect(test_frame)
    print(f"Detected {len(detections)} objects")
    
    for det in detections:
        print(f"  - {det.class_name}: {det.confidence:.2f}")
    
    # Visualize detections
    vis_det = draw_detections(test_frame, detections, CLASS_NAMES)
    
    # Display
    plt.figure(figsize=(15, 10))
    plt.imshow(cv2.cvtColor(vis_det, cv2.COLOR_BGR2RGB))
    plt.title('Detection Results')
    plt.axis('off')
    plt.show()
else:
    print(f"Error: Could not read from {INPUT_PATH}")

## 5. Run Full Tracking Pipeline

In [ ]:
from core.pipeline import HockeyTrackingPipeline, VideoReader, VideoWriter
from tqdm.notebook import tqdm
import time

# Create output directory
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# Reset tracker
Track.reset_id()
tracker = McByteTracker(
    track_thresh=config.tracker.track_thresh,
    track_buffer=config.tracker.track_buffer,
    match_thresh=config.tracker.match_thresh,
    low_thresh=config.tracker.low_thresh,
    new_track_thresh=config.tracker.new_track_thresh,
    frame_rate=config.tracker.frame_rate,
    use_gmc=config.tracker.use_gmc,
)

visualizer.reset_histories()

In [ ]:
# Run tracking
reader = VideoReader(INPUT_PATH)
writer = VideoWriter(OUTPUT_PATH, reader.width, reader.height, reader.fps)

results = []
frame_times = []

print(f"Processing {len(reader)} frames...")

for frame_id, frame in tqdm(reader, total=len(reader), desc="Tracking"):
    start = time.time()
    
    # Detect
    detections = detector.detect(frame)
    
    # Track
    tracks = tracker.update(detections, frame)
    
    # Visualize
    vis_frame = visualizer.draw_tracks(frame, tracks)
    
    # Save
    writer.write(vis_frame)
    
    # Record
    frame_time = time.time() - start
    frame_times.append(frame_time)
    results.append({
        'frame_id': frame_id,
        'num_tracks': len(tracks),
        'tracks': [(t.track_id, t.class_id, t.tlwh.tolist()) for t in tracks]
    })

reader.release()
writer.release()

# Summary
avg_fps = 1.0 / np.mean(frame_times)
print(f"\n{'='*50}")
print(f"Processing complete!")
print(f"Average FPS: {avg_fps:.1f}")
print(f"Output saved to: {OUTPUT_PATH}")

## 6. Analyze Results

In [ ]:
# Track statistics
track_counts = [r['num_tracks'] for r in results]

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(track_counts)
plt.xlabel('Frame')
plt.ylabel('Number of Tracks')
plt.title('Active Tracks Over Time')
plt.grid(True)

plt.subplot(1, 2, 2)
fps_values = [1.0/t for t in frame_times]
plt.plot(fps_values, alpha=0.5)
plt.axhline(y=np.mean(fps_values), color='r', linestyle='--', label=f'Avg: {np.mean(fps_values):.1f} FPS')
plt.xlabel('Frame')
plt.ylabel('FPS')
plt.title('Processing Speed')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\nStatistics:")
print(f"  Average tracks per frame: {np.mean(track_counts):.1f}")
print(f"  Max tracks in frame: {max(track_counts)}")
print(f"  Processing speed: {np.mean(fps_values):.1f} FPS")

## 7. Export MOT Format Results

In [ ]:
# Save in MOT format
mot_output = "outputs/tracking_results.txt"

with open(mot_output, 'w') as f:
    for r in results:
        for track_id, class_id, tlwh in r['tracks']:
            x, y, w, h = tlwh
            line = f"{r['frame_id']},{track_id},{x:.2f},{y:.2f},{w:.2f},{h:.2f},1,{class_id},-1,-1\n"
            f.write(line)

print(f"MOT results saved to: {mot_output}")

## 8. Using the High-Level Pipeline API

For simpler usage, use the `HockeyTrackingPipeline` class.

In [ ]:
# Alternative: Use the high-level pipeline
# This handles everything automatically

# from core.pipeline import HockeyTrackingPipeline
#
# pipeline = HockeyTrackingPipeline(config)
# results = pipeline.run(
#     INPUT_PATH,
#     output_path="outputs/tracked_output.mp4",
#     show_preview=False,
#     use_masks=USE_MASKS,
# )
#
# # Save MOT format
# pipeline.save_mot_results(results, "outputs/mot_results.txt")

## 9. Display Output Video

In [ ]:
# Display the output video (if running in Jupyter)
if os.path.exists(OUTPUT_PATH):
    try:
        from IPython.display import Video
        display(Video(OUTPUT_PATH, embed=True, width=800))
    except:
        print(f"Video saved at: {OUTPUT_PATH}")
        print("Open it with your video player to view.")

---

## 🔧 Troubleshooting

### Common Issues

1. **CUDA out of memory**
   - Reduce `max_internal_size` in mask config
   - Set `half_precision = True`
   - Process at lower resolution

2. **Tracking IDs jumping around**
   - Increase `track_buffer` for longer re-identification window
   - Lower `match_thresh` for easier association
   - Enable GMC for camera motion handling

3. **Missing detections**
   - Lower `confidence_threshold`
   - Check if your YOLO model is trained properly

4. **Slow processing**
   - Disable mask propagation (`USE_MASKS = False`)
   - Use smaller input resolution
   - Ensure GPU is being used

### Performance Tips

- Use FP16 on supported GPUs
- Lower mask `max_internal_size` for faster mask propagation
- Adjust `mem_every` to reduce memory updates
- Consider frame skipping for real-time applications